# OurDataset Dose Generation Notebook

It loads one or many patient folders, computes PyDoseRT dose, and saves:
- `data/<patient>/pydosert_dose/dose_pred.npy`
- optional QC figures under `out/ourdataset_notebook_qc/`
- a summary CSV under `out/`

After this notebook finishes, use `examples/plotting_from_patient.ipynb` for publication plots.



In [ ]:
%matplotlib inline

from pathlib import Path
import sys
import time
import traceback

import numpy as np
import pandas as pd
import torch
from scipy.ndimage import binary_fill_holes, binary_erosion

def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src").exists():
            return candidate
    raise RuntimeError("Could not locate repo root (missing pyproject.toml/src).")

repo_root = find_repo_root(Path.cwd())
src_path = repo_root / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from pydosert import DoseEngine
from pydosert.data import MachineConfig, OptimizationConfig, loaders
from pydosert.objectives.metrics import result_validation
from pydosert.utils.plotting import quick_plot, print_comparison_plot
from pydosert.utils.utils import find_patient_paths

print(f"Repo root: {repo_root}")



In [ ]:
# Optional helper: list available patients under data/
data_root = repo_root / "data"
patients = sorted([p.name for p in data_root.iterdir() if p.is_dir()]) if data_root.exists() else []
print(f"Found {len(patients)} patient folders")
print(patients)



In [ ]:
# -------------------------------
# User controls (edit this cell)
# -------------------------------
RUN_ALL_PATIENTS = False
PATIENT_DIR = repo_root / "data" / "00LGN1CKZ"
DATA_ROOT = repo_root / "data"

STRUCT_NAMES = ["PTV", "Body"]
NEW_SPACING = (3.0, 3.0, 3.0)
USE_DELIVERY = True
CROP_VOLUME = False

KERNEL_SIZE = 51
MACHINE_PRESET = repo_root / "src" / "pydosert" / "data" / "machine_presets" / "ourdataset_10MV.json"
MLC_TRANSMISSION = 0.0
OPTIMIZATION_JSON = repo_root / "src" / "pydosert" / "data" / "optimization_presets" / "ourdataset.json"

SAVE_PRED_TO_PATIENT = True
SAVE_QC_FIGURES = False
RUN_VALIDATION = True
SKIP_IF_PRED_EXISTS = True
FORCE_RECOMPUTE = False

GAMMA_THRESHOLD_DISTANCE = 3.0
GAMMA_THRESHOLD_DOSE = 3.0

RESULTS_CSV = repo_root / "out" / "ourdataset_results_summary.csv"
ERRORS_CSV = repo_root / "out" / "ourdataset_errors.csv"
QC_OUT_DIR = repo_root / "out" / "ourdataset_notebook_qc"

print(f"PATIENT_DIR: {PATIENT_DIR}")
print(f"RUN_ALL_PATIENTS: {RUN_ALL_PATIENTS}")
print(f"MACHINE_PRESET: {MACHINE_PRESET}")
print(f"OPTIMIZATION_JSON: {OPTIMIZATION_JSON}")



In [ ]:
# Fallback to Vienna presets if OurDataset presets are not present
if not MACHINE_PRESET.exists():
    MACHINE_PRESET = repo_root / "src" / "pydosert" / "data" / "machine_presets" / "vienna_10MV.json"
if not OPTIMIZATION_JSON.exists():
    OPTIMIZATION_JSON = repo_root / "src" / "pydosert" / "data" / "optimization_presets" / "vienna.json"
machine_config = MachineConfig(preset=str(MACHINE_PRESET), mlc_transmission=MLC_TRANSMISSION)
optimization = OptimizationConfig.from_json(str(OPTIMIZATION_JSON))

def resolve_patient_dirs() -> list[Path]:
    if RUN_ALL_PATIENTS:
        if not DATA_ROOT.exists():
            raise FileNotFoundError(f"Data root not found: {DATA_ROOT}")
        return sorted([p for p in DATA_ROOT.iterdir() if p.is_dir()])

    candidate = PATIENT_DIR if PATIENT_DIR.is_absolute() else (repo_root / PATIENT_DIR)
    candidate = candidate.resolve()
    if not candidate.exists():
        raise FileNotFoundError(f"Patient directory not found: {candidate}")
    return [candidate]

def choose_ptv_name(struct_names: list[str]) -> str | None:
    for name in struct_names:
        if "ptv" in name.lower():
            return name
    return None

def process_patient(patient_dir: Path, device: torch.device, dtype: torch.dtype) -> dict:
    patient_name = patient_dir.name
    ct_folder, rtplan_path, rtdose_path, rtstruct_path = find_patient_paths(patient_dir)

    patient, beam_sequences = loaders.load_dicom(
        ct_folder=ct_folder,
        dose_path=rtdose_path,
        plan_path=rtplan_path,
        struct_path=rtstruct_path,
        new_spacing=NEW_SPACING,
        struct_names=STRUCT_NAMES,
        use_delivery=USE_DELIVERY,
        crop_volume=CROP_VOLUME,
    )

    body_key = next((k for k in patient.structures.keys() if k.lower() == "body" or "body" in k.lower()), None)
    if body_key is not None:
        patient.dose = patient.dose * patient.structures[body_key]
    else:
        print(f"[{patient_name}] Warning: no Body structure found, using full volume mask.")

    beam_sequence = beam_sequences[0].to(device).to(dtype)
    patient = patient.to(device).to(dtype)
    dose_volume = patient.dose

    if body_key is not None:
        body_mask = patient.structures[body_key]
    else:
        body_mask = torch.ones_like(patient.dose, dtype=torch.bool)

    density_image = torch.where(body_mask, patient.density_image, torch.zeros_like(patient.density_image))

    dose_engine = DoseEngine(
        kernel_size=KERNEL_SIZE,
        dose_grid_spacing=patient.resolution,
        dose_grid_shape=density_image.shape,
        machine_config=machine_config,
        beam_template=beam_sequence,
        device=device,
        dtype=dtype,
    )

    dose_engine.calibrate(
        calibration_mu=machine_config.calibration_mu,
        original_beam_template=beam_sequence,
    )

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    t0 = time.time()
    dose_pred_4d = dose_engine.compute_dose_sequential(beam_sequence, density_image=density_image).detach()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed_s = time.time() - t0

    dose_pred = torch.where(body_mask, dose_pred_4d[0], torch.zeros_like(dose_volume))

    dose_output_path = patient_dir / "pydosert_dose" / "dose_pred.npy"
    if SAVE_PRED_TO_PATIENT:
        dose_output_path.parent.mkdir(parents=True, exist_ok=True)
        np.save(dose_output_path, dose_pred.detach().cpu().numpy())

    body_np = body_mask.detach().cpu().numpy().astype(bool)
    ext_mask = binary_erosion(binary_fill_holes(body_np), np.ones((3, 3, 3)), iterations=7)
    dose_np = dose_volume.detach().cpu().numpy()
    ext_mask &= (dose_np > 0.1 * float(dose_volume.max().item()))
    if not ext_mask.any():
        ext_mask = body_np

    mae_map = torch.abs(dose_pred - dose_volume).detach().cpu().numpy()
    mae_cgy = float(np.mean(100.0 * mae_map[ext_mask]))

    ptv_name = choose_ptv_name(list(patient.structures.keys()))
    scale = np.nan
    if ptv_name is not None:
        ptv_mask = patient.structures[ptv_name] > 0
        pred_mean = dose_pred[ptv_mask].mean()
        if torch.isfinite(pred_mean) and float(pred_mean.item()) != 0.0:
            scale = float((dose_volume[ptv_mask].mean() / pred_mean).item())

    row = {
        "patient_name": patient_name,
        "dose_pred_path": str(dose_output_path),
        "dose_compute_seconds": round(elapsed_s, 2),
        "mae_cgy": round(mae_cgy, 3),
        "ptv_scale_ratio_ref_over_pred": scale,
    }

    if RUN_VALIDATION:
        validation = result_validation(
            patient,
            machine_config,
            beam_sequence,
            dose_pred,
            optimization,
            compute_gamma=True,
            compute_clinical_criteria=True,
            global_normalisation=None,
            gamma_threshold_distance=GAMMA_THRESHOLD_DISTANCE,
            gamma_threshold_dose=GAMMA_THRESHOLD_DOSE,
        )
        for key, value in validation.items():
            if isinstance(value, (int, float, np.floating, np.integer, bool)):
                row[key] = value

    if SAVE_QC_FIGURES:
        QC_OUT_DIR.mkdir(parents=True, exist_ok=True)
        quick_out = QC_OUT_DIR / f"quick_{patient_name}.png"
        comparison_out = QC_OUT_DIR / f"comparison_{patient_name}.png"
        quick_plot(patient, dose_pred, title=f"{patient_name} | MAE {mae_cgy:.2f} cGy", out_path=str(quick_out))
        print_comparison_plot(optimization, patient, dose_pred, out_path=str(comparison_out))

    del dose_engine, patient, beam_sequence, dose_pred_4d, dose_pred, dose_volume
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return row





In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dtype = torch.float32
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"CUDA GPU: {torch.cuda.get_device_name(0)}")

patient_dirs = resolve_patient_dirs()
print(f"Will process {len(patient_dirs)} patient folder(s)")

rows = []
errors = []

for patient_dir in patient_dirs:
    patient_name = patient_dir.name
    pred_path = patient_dir / "pydosert_dose" / "dose_pred.npy"

    if SKIP_IF_PRED_EXISTS and pred_path.exists() and not FORCE_RECOMPUTE:
        print(f"[{patient_name}] Skip (pred dose already exists): {pred_path}")
        continue

    print(f"[{patient_name}] Processing...")
    try:
        row = process_patient(patient_dir, device, dtype)
        rows.append(row)
        gamma = row.get("gamma_pass_rate", np.nan)
        gamma_ok = isinstance(gamma, (int, float, np.floating, np.integer)) and np.isfinite(gamma)
        gamma_text = f" | gamma {gamma:.3f}" if gamma_ok else ""
        compute_s = row["dose_compute_seconds"]
        mae_cgy = row["mae_cgy"]
        print(f"[{patient_name}] Done in {compute_s} s | MAE {mae_cgy} cGy{gamma_text}")
    except Exception as exc:
        msg = f"{type(exc).__name__}: {exc}"
        errors.append({"patient_name": patient_name, "error": msg})
        print(f"[{patient_name}] ERROR: {msg}")
        traceback.print_exc()

if rows:
    RESULTS_CSV.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(rows).to_csv(RESULTS_CSV, index=False)
    print(f"Saved results CSV: {RESULTS_CSV}")
    display(pd.DataFrame(rows))
else:
    print("No new patients processed.")

if errors:
    ERRORS_CSV.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(errors).to_csv(ERRORS_CSV, index=False)
    print(f"Saved errors CSV: {ERRORS_CSV}")
    display(pd.DataFrame(errors))

print("Next step: open examples/plotting_from_patient.ipynb and set PATIENT_DIR to one processed patient folder.")



## How To Use With Plotting Notebook

1. Run this notebook to generate `pydosert_dose/dose_pred.npy`.
2. Open `examples/plotting_from_patient.ipynb`.
3. Set `PATIENT_DIR` to the same patient folder.
4. Run plotting cells to create publication figures.

